# 데모 5 — Deep Agents: 그래프를 부하 직원으로 부리는 에이전트

**한 줄 요약**: `create_deep_agent` 로 리서치 에이전트를 만들고,
**데모 1의 Adaptive RAG 그래프를 서브에이전트로** 등록해
"플래닝(TODO) → 위임 → 웹 조사 → 파일 산출 → 비평 반영" 전 과정을 시연한다.

```mermaid
flowchart TB
    U([유저: 브리프 작성 요청]) --> MA["메인 딥 에이전트"]
    MA -- "write_todos" --> T["📋 TODO (플래닝)"]
    MA -- "task(subagent_type=...)" --> RA["rag-agent<br/>= 데모 1 Adaptive RAG 그래프!"]
    MA -- "task(subagent_type=...)" --> CA["critique-agent<br/>(초안 비평)"]
    MA -- "internet_search" --> W["🌐 웹 (Tavily/모의)"]
    MA -- "write_file" --> F["📄 outputs/deepagent_work/<br/>loop_engineering_brief.md"]
    style RA fill:#d3f9d8
    style MA fill:#d0ebff
```

**딥 에이전트 4요소와 이 데모의 대응**

| 요소 | 이 데모에서 |
|---|---|
| 플래닝 도구 | `write_todos` — 계획이 상태에 남아 장기 작업에도 목표를 안 잃음 |
| 서브에이전트 | `critique-agent`(프롬프트 정의) + `rag-agent`(**컴파일된 LangGraph 그래프**) |
| 파일시스템 | `write_file` → 실제 디스크(`outputs/deepagent_work/`)에 산출물 |
| 상세 프롬프트 | 리서치 절차·품질 기준을 명시한 system_prompt |

> **재사용의 정점**: deepagents 의 `CompiledSubAgent` 는 `{"name", "description",
> "runnable"}` 형태로 **이미 컴파일된 그래프**를 서브에이전트로 받는다 (공식 지원).
> 단, runnable 의 상태 스키마에 `messages` 키가 있어야 하므로 데모 1 그래프를
> 얇은 브리지 그래프로 감싼다 — 아래 셀에서 6줄이면 된다.

**mock 모드**: 메인 에이전트의 판단 시퀀스를 `ScriptedChatModel` 로 재생한다.
도구 실행·서브에이전트(그래프 포함)·파일 쓰기는 전부 **진짜로** 동작한다 —
대본이 있는 것은 "말"뿐이고 "행동"은 실제다.

In [ ]:
# ── 준비: 데모 1 그래프를 messages 브리지로 감싸기 ────────────
import os, uuid
from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

from common import OUTPUTS
from common.console import banner, step, console
from common.llm import ScriptedChatModel, get_llm, provider
from common.viz import show_graph
from demo1_workflow_to_loop.adaptive_rag import build_adaptive_rag

banner("데모 5 — Deep Agents", f"LLM_PROVIDER = {provider()}")

rag_app = build_adaptive_rag()          # 데모 1 완성형 그래프

class BridgeState(TypedDict):
    messages: Annotated[list, add_messages]   # CompiledSubAgent 요건: messages 키

def call_adaptive_rag(state: BridgeState) -> dict:
    """task 도구가 넘긴 description(마지막 메시지)을 질문으로 그래프 실행."""
    question = state["messages"][-1].content
    result = rag_app.invoke({"question": question, "rewrite_count": 0})
    return {"messages": [AIMessage(content=result["answer"])]}

_b = StateGraph(BridgeState)
_b.add_node("adaptive_rag", call_adaptive_rag)
_b.add_edge(START, "adaptive_rag")
_b.add_edge("adaptive_rag", END)
rag_bridge = _b.compile()               # ← 이것이 서브에이전트로 들어간다
print("rag-agent 브리지 준비 완료 (데모 1 그래프 재사용)")

In [ ]:
# ── 도구: 웹 검색 (Tavily 키 없으면 모의 결과 자동 폴백) ───────
@tool
def internet_search(query: str) -> str:
    """웹에서 최신 정보를 검색해 요약 텍스트를 반환한다."""
    from common.websearch import format_web_results, web_search
    return format_web_results(web_search(query))

## mock 대본 (ScriptedChatModel)

메인 에이전트 LLM 이 내릴 판단 7수를 미리 적어둔다.
실 모델(openai/ollama)에서는 이 대본 대신 모델이 실시간으로 판단한다.

In [ ]:
def _tc(name: str, args: dict) -> dict:
    return {"name": name, "args": args, "id": f"call_{uuid.uuid4().hex[:6]}", "type": "tool_call"}

TASK = "우리 코퍼스와 웹을 조사해 'Loop Engineering이 왜 중요한가' 1페이지 브리프를 작성해줘"
DRAFT_FILE = "loop_engineering_brief_draft.md"   # 초안
BRIEF = "loop_engineering_brief.md"              # 최종본
# 주의: deepagents 의 write_file 은 기존 파일을 덮어쓰지 않는다(수정은 edit_file).
# 그래서 초안과 최종본을 다른 파일로 산출한다 — 개정 이력이 남아 데모에도 좋다.

DRAFT = """# Loop Engineering이 왜 중요한가 (초안)

LLM 출력은 확률적이라 단발 실행의 품질을 보장할 수 없다.
루프 엔지니어링은 중간 결과를 평가하고 부족하면 되돌아가는 구조로
품질의 하한을 끌어올리는 설계 방법론이다.

웹 동향: 커뮤니티는 워크플로보다 루프와 평가 사이클을 먼저 설계하라고 조언한다.
"""

FINAL = """# Loop Engineering이 왜 중요한가

## 요지
LLM 출력은 확률적이므로 단발 실행의 품질을 보장할 수 없다. 루프 엔지니어링은
중간 결과를 스스로 평가하고 부족하면 이전 단계로 되돌아가는 구조로
**품질의 하한**을 끌어올리는 설계 방법론이다.

## 근거 (코퍼스 「루프 엔지니어링」 문서)
- 실패가 사용자에게 전달되는 대신 감지되고 복구된다 — 반례: 루프 없는 직선
  파이프라인은 검색이 빗나가도 그대로 오답을 내보낸다.
- 루프에는 반드시 탈출 조건(최대 반복·타임아웃)이 필요하다.
- 런타임 루프(재작성·재시도)와 개발 루프(관측→평가→개선) 두 층위가 함께 돌아야 한다.

## 웹 동향
에이전트 생태계는 루프 기반 설계, 휴먼 인 더 루프, 관측성 표준(OTel) 중심으로
진화 중이며, "워크플로보다 루프·평가 사이클을 먼저 설계하라"는 조언이 많다.

## 다음 단계 3가지 (비평 반영)
1. 파일럿 그래프에 관련성 평가 + 쿼리 재작성 루프 도입 (탈출 조건 max 2회)
2. 발행류 행동 앞에 interrupt 승인 게이트 추가
3. Langfuse 데이터셋으로 실패 사례 회귀 테스트 상시화
"""

MAIN_SCRIPT = [
    AIMessage(content="", tool_calls=[_tc("write_todos", {"todos": [
        {"content": "코퍼스에서 루프 엔지니어링 정의·중요성 조사 (rag-agent 위임)", "status": "in_progress"},
        {"content": "웹에서 최신 동향 검색", "status": "pending"},
        {"content": "1페이지 브리프 초안 작성", "status": "pending"},
        {"content": "critique-agent 비평 받아 반영", "status": "pending"},
    ]})]),
    AIMessage(content="", tool_calls=[_tc("task", {
        "description": "루프 엔지니어링이란 무엇이고 왜 중요한가? 내부 용어집 근거로 정리해줘",
        "subagent_type": "rag-agent"})]),                      # ← 데모 1 그래프 호출!
    AIMessage(content="", tool_calls=[_tc("internet_search", {
        "query": "Loop Engineering 에이전트 설계 최신 동향"})]),
    AIMessage(content="", tool_calls=[_tc("write_file", {
        "file_path": DRAFT_FILE, "content": DRAFT})]),
    AIMessage(content="", tool_calls=[_tc("task", {
        "description": f"{DRAFT_FILE} 초안이다. 부족한 점을 비평해줘:\n\n" + DRAFT,
        "subagent_type": "critique-agent"})]),
    AIMessage(content="", tool_calls=[_tc("write_file", {
        "file_path": BRIEF, "content": FINAL})]),
    AIMessage(content="", tool_calls=[_tc("write_todos", {"todos": [
        {"content": "코퍼스에서 루프 엔지니어링 정의·중요성 조사 (rag-agent 위임)", "status": "completed"},
        {"content": "웹에서 최신 동향 검색", "status": "completed"},
        {"content": "1페이지 브리프 초안 작성", "status": "completed"},
        {"content": "critique-agent 비평 받아 반영", "status": "completed"},
    ]})]),
    AIMessage(content=f"브리프 작성을 완료했습니다 → {BRIEF} (비평 반영: 근거 인용·반례·다음 단계 추가)"),
]

In [ ]:
# ── 딥 에이전트 조립 ─────────────────────────────────────────
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

import shutil

WORKDIR = OUTPUTS / "deepagent_work"
shutil.rmtree(WORKDIR, ignore_errors=True)   # 재실행 멱등성: 이전 산출물 정리
WORKDIR.mkdir(parents=True, exist_ok=True)

if provider() == "mock":
    main_model = ScriptedChatModel(script=MAIN_SCRIPT)
    critique_model = ScriptedChatModel(script=[AIMessage(content=get_llm("critique").invoke("비평").content)])
else:
    main_model = get_llm()          # 실 모델: 도구 호출 판단을 모델이 직접
    critique_model = get_llm()

agent = create_deep_agent(
    model=main_model,
    tools=[internet_search],
    system_prompt=(
        "너는 리서치 에이전트다. 절차: ① write_todos 로 계획 수립 ② 내부 지식은 "
        "rag-agent 에, 비평은 critique-agent 에 task 로 위임 ③ 웹 최신 동향은 "
        "internet_search ④ 산출물은 write_file 로 저장 ⑤ 비평을 반영해 최종본을 만든다. "
        "모든 산출물은 한국어로, 근거를 인용해 작성한다."
    ),
    subagents=[
        {   # 프롬프트로 정의한 서브에이전트
            "name": "critique-agent",
            "description": "초안의 논리·근거·구성을 비평하는 리뷰어",
            "system_prompt": "주어진 초안의 약점을 3가지 이내로 구체적으로 비평하라.",
            "model": critique_model,
            "tools": [],
        },
        {   # ★ 컴파일된 LangGraph 그래프를 그대로 서브에이전트로 (CompiledSubAgent)
            "name": "rag-agent",
            "description": "내부 용어집(코퍼스) 질의응답 — 데모 1 Adaptive RAG 그래프",
            "runnable": rag_bridge,
        },
    ],
    backend=FilesystemBackend(root_dir=WORKDIR, virtual_mode=True),  # 실제 디스크 산출
)
show_graph(agent, "deepagent")

In [ ]:
# ── Langfuse 트레이싱 (선택): 플래닝·서브에이전트가 한 트리로 ──
import httpx
from common.tracing import get_langfuse_handler, langfuse_enabled, trace_config

run_config = {"recursion_limit": 60}
BASE = os.getenv("LANGFUSE_BASE_URL") or "http://localhost:3000"
try:
    lf_up = langfuse_enabled() and httpx.get(f"{BASE}/api/public/health", timeout=3).is_success
except Exception:
    lf_up = False
if lf_up:
    run_config.update(trace_config(get_langfuse_handler(),
                                   session_id="demo5", user_id="presenter", tags=["demo5"]))
    print(f"Langfuse 트레이싱 ON → {BASE} (플래닝→task→서브에이전트 중첩 트리 확인)")
else:
    print("Langfuse 미접속 — 트레이싱 없이 진행 (make up 후 다시 실행하면 트레이스가 남는다)")

In [ ]:
# ── 실행! ────────────────────────────────────────────────────
step("딥 에이전트 실행")
result = agent.invoke({"messages": [{"role": "user", "content": TASK}]}, config=run_config)

for m in result["messages"]:
    kind = m.__class__.__name__
    if getattr(m, "tool_calls", None):
        for c in m.tool_calls:
            arg = c["args"].get("subagent_type") or c["args"].get("query") \
                  or c["args"].get("file_path") or ""
            console.print(f"  [bold cyan]🤖 {c['name']}[/] [dim]{arg}[/]")
    elif isinstance(m, ToolMessage):
        one = str(m.content).replace("\n", " ")[:76]
        console.print(f"     [green]↳[/] [dim]{one}[/]")
    elif isinstance(m, (HumanMessage,)) :
        console.print(f"  [bold]🙋 {str(m.content)[:60]}[/]")
    elif kind == "AIMessage" and m.content:
        console.print(f"  [bold magenta]🧠 {str(m.content)[:76]}[/]")

In [ ]:
# ── 산출물 확인: 계획(TODO) 상태 + 실제 파일 ─────────────────
step("산출물")
for todo in result.get("todos", []):
    mark = {"completed": "✅", "in_progress": "🔄"}.get(todo.get("status"), "⬜")
    console.print(f"  {mark} {todo.get('content')}")

brief_path = WORKDIR / BRIEF
assert brief_path.exists(), "브리프 파일이 생성되어야 합니다"
print(f"\n── {brief_path.relative_to(OUTPUTS.parent)} ──\n")
print(brief_path.read_text(encoding="utf-8"))
if lf_up:
    print(f"\n트레이스: {BASE} → Sessions=demo5 에서 중첩 트리 확인")

## 정리 — 발표 포인트

1. **자산 재사용의 정점**: 데모 1에서 만든 그래프가 코드 6줄(브리지)로 딥 에이전트의
   부하 직원이 됐다. 그래프 → 도구 → 서브에이전트, 모두 같은 LangGraph 생태계 안의 부품이다.
2. **컨텍스트 격리**: rag-agent 의 검색·재작성 과정(수천 토큰)은 메인 에이전트 컨텍스트에
   들어오지 않는다 — 돌아오는 것은 요약된 결과 한 줄. 이것이 긴 작업에서 표류를 막는다.
3. **관측 없이는 디버깅 불가**: 플래닝→위임→종합의 깊은 트리는 Langfuse 트레이스로 봐야
   한다 (데모 3의 인프라가 그대로 재사용된다는 점을 강조).

**세션 마무리로**: selfhost/README.md 의 사내망 체크리스트를 띄우고
"오늘 본 전부(그래프·HITL·관측·평가·딥 에이전트)가 로컬 자급자족 스택 위에서 돌았다"로 클로징.